In [1]:
import os
import cv2
import random
import joblib
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tqdm import tqdm
import time

In [2]:
print("Loading pre-extracted HOG features from disk...")

# Make sure this filename matches the one from your extraction script!
data_package = joblib.load("final_hog_features.joblib")

# Unpack the arrays
X = data_package["X"]
y = data_package["y"]

# Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data loaded and split successfully!")
print(f"Total Images: {len(y)}")
print(f"Training Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")

Loading pre-extracted HOG features from disk...
Data loaded and split successfully!
Total Images: 14500
Training Data Shape: (11600, 1764)
Testing Data Shape: (2900, 1764)


In [3]:
print("Training Proposed Model (StandardScaler -> PCA -> SVM)...")
start_time = time.time()

# Build the pipeline
proposed_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)), 
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale')) 
])

# Train the model
proposed_pipeline.fit(X_train, y_train)

# Make predictions on the test set
predictions = proposed_pipeline.predict(X_test)

print(f"Training complete in {time.time() - start_time:.2f} seconds!")
print("Ready for evaluation.")

Training Proposed Model (StandardScaler -> PCA -> SVM)...
Training complete in 75.00 seconds!
Ready for evaluation.


In [4]:
# Calculate metrics
accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, average='weighted')

print("="*50)
print(" FINAL MODEL EVALUATION (HOG-SVM)")
print("="*50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("="*50)

# Check against project criteria
if f1 >= 0.85:
    print(f"\nSUCCESS: The model hit the >= 0.85 F1-Score criteria with a score of {f1:.4f}!")
else:
    print(f"\nNOTE: The model achieved {f1:.4f}. You may need to tune the SVM parameters (C and gamma) to hit 0.85.")

print("\nDetailed Classification Report:")
print(classification_report(y_test, predictions))

 FINAL MODEL EVALUATION (HOG-SVM)
Accuracy:  0.8659
F1-Score:  0.8660

SUCCESS: The model hit the >= 0.85 F1-Score criteria with a score of 0.8660!

Detailed Classification Report:
              precision    recall  f1-score   support

           A       0.79      0.91      0.85       110
           B       0.84      0.79      0.81        96
           C       0.95      0.93      0.94       105
           D       0.87      0.83      0.85        96
         DEL       0.98      0.96      0.97       112
           E       0.75      0.83      0.79        96
           F       0.88      0.91      0.89       102
           G       0.92      0.87      0.89        89
           H       0.94      0.95      0.94       111
           I       0.93      0.85      0.89        99
           J       0.93      0.98      0.95        92
           K       0.72      0.88      0.79        97
           L       0.93      0.96      0.95        90
           M       0.77      0.80      0.79        94
        

In [5]:
# Assuming you have already run:
# proposed_pipeline.fit(X_train, y_train)

print("Saving the trained model...")

# Save the entire pipeline to a file named 'asl_svm_model.joblib'
# compress=9 ensures the file is small enough to bypass GitHub's 25MB limit
joblib.dump(proposed_pipeline, 'asl_svm_model.joblib', compress=9)

print("Model saved successfully as 'asl_svm_model.joblib'!")

Saving the trained model...
Model saved successfully as 'asl_svm_model.joblib'!
